In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn kagglehub[pandas-datasets]
%pip install kagglehub[pandas-datasets]

In [ ]:
import numpy as np

class MLP:
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):

        self.weights_input_hidden1 = np.random.randn(input_size, hidden_size1) * np.sqrt(2 / input_size)
        self.bias_hidden1 = np.zeros((1, hidden_size1))
        
        self.weights_hidden1_hidden2 = np.random.randn(hidden_size1, hidden_size2) * np.sqrt(2 / hidden_size1)
        self.bias_hidden2 = np.zeros((1, hidden_size2))
        
        self.weights_hidden2_hidden3 = np.random.randn(hidden_size2, hidden_size3) * np.sqrt(2 / hidden_size2)
        self.bias_hidden3 = np.zeros((1, hidden_size3))
        
        self.weights_hidden_output = np.random.randn(hidden_size3, output_size) * np.sqrt(2 / hidden_size3)
        self.bias_output = np.zeros((1, output_size))

    def relu(self, x):
        return np.maximum(0, x)

    def relu_derivative(self, x):
        return (x > 0).astype(float)

    def softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / exp_x.sum(axis=1, keepdims=True)

    def forward(self, X):
        self.hidden_input1 = np.dot(X, self.weights_input_hidden1) + self.bias_hidden1
        self.hidden_output1 = self.relu(self.hidden_input1)

        self.hidden_input2 = np.dot(self.hidden_output1, self.weights_hidden1_hidden2) + self.bias_hidden2
        self.hidden_output2 = self.relu(self.hidden_input2)

        self.hidden_input3 = np.dot(self.hidden_output2, self.weights_hidden2_hidden3) + self.bias_hidden3
        self.hidden_output3 = self.relu(self.hidden_input3)

        self.final_input = np.dot(self.hidden_output3, self.weights_hidden_output) + self.bias_output
        self.final_output = self.softmax(self.final_input)
        return self.final_output

    def backward(self, X, y, output, lr):
        output_error = output - y

        hidden_error3 = np.dot(output_error, self.weights_hidden_output.T) * self.relu_derivative(self.hidden_input3)
        hidden_error2 = np.dot(hidden_error3, self.weights_hidden2_hidden3.T) * self.relu_derivative(self.hidden_input2)
        hidden_error1 = np.dot(hidden_error2, self.weights_hidden1_hidden2.T) * self.relu_derivative(self.hidden_input1)

        self.weights_hidden_output -= lr * np.dot(self.hidden_output3.T, output_error)
        self.bias_output -= lr * np.sum(output_error, axis=0, keepdims=True)

        self.weights_hidden2_hidden3 -= lr * np.dot(self.hidden_output2.T, hidden_error3)
        self.bias_hidden3 -= lr * np.sum(hidden_error3, axis=0, keepdims=True)

        self.weights_hidden1_hidden2 -= lr * np.dot(self.hidden_output1.T, hidden_error2)
        self.bias_hidden2 -= lr * np.sum(hidden_error2, axis=0, keepdims=True)

        self.weights_input_hidden1 -= lr * np.dot(X.T, hidden_error1)
        self.bias_hidden1 -= lr * np.sum(hidden_error1, axis=0, keepdims=True)

    def train(self, X, y, epochs, lr):
        for epoch in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output, lr)
            if (epoch + 1) % 100 == 0:
                loss = -np.sum(y * np.log(output + 1e-9)) / X.shape[0]
                print(f"Epoch {epoch + 1}, Loss: {loss:.4f}")

    def predict(self, X):
        output = self.forward(X)
        return np.argmax(output, axis=1)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import numpy as np
import pandas as pd

def run_experiment(df, hidden_size=121, epochs=7200, lr=0.0002, random_state=42):
    """
    Runs a single experiment and returns results
    """
    # Prepare data
    X = df.drop('quality', axis=1).to_numpy()
    y = df['quality'].to_numpy()
    
    # Split data into train, validation, test sets
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=random_state
    )
    
    print(f"Training set size: {X_train.shape[0]}")
    print(f"Validation set size: {X_val.shape[0]}")
    print(f"Test set size: {X_test.shape[0]}")
    
    # One-hot encode labels
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))                                           
    y_val_oh = encoder.transform(y_val.reshape(-1, 1))
    y_test_oh = encoder.transform(y_test.reshape(-1, 1))
    
    # Initialize and train model
    mlp = MLP(
        input_size=X_train.shape[1],
        hidden_size1=64,
        hidden_size2=32,
        hidden_size3=16,
        output_size=y_train_oh.shape[1]
    )

    mlp.train(X_train, y_train_oh, epochs, lr)
    
    # Evaluate model
    y_val_pred = mlp.predict(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    
    y_test_pred = mlp.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)
    
    print(f"Validation accuracy: {val_accuracy:.4f}")
    print(f"Test accuracy: {test_accuracy:.4f}")
    
    return {
        'model': mlp,
        'val_accuracy': val_accuracy,
        'test_accuracy': test_accuracy,
        'confusion_matrix': cm,
        'y_test': y_test,
        'y_pred': y_test_pred,
        'classes': np.unique(y)
    }

# Load dataset
df = pd.read_csv("C:/Users/krzys/Downloads/WSI_z5_data.csv")

# Show dataset info
print("Dataset info:")
print(df.info())

# Check class distribution
print("\nClass distribution:")
class_distribution = df['quality'].value_counts().sort_index()
print(class_distribution)

# Number of experiments
num_experiments = 5
results = []

# Run multiple experiments
for i in range(num_experiments):
    print(f"\n\n=== Experiment {i+1}/{num_experiments} ===")
    result = run_experiment(df, random_state=42+i)
    results.append(result)

# Collect results
val_accuracies = [res['val_accuracy'] for res in results]
test_accuracies = [res['test_accuracy'] for res in results]
confusion_matrices = [res['confusion_matrix'] for res in results]

# Accuracy statistics
print("\n=== Results statistics ===")
print("Validation accuracy:")
print(f"  Mean: {np.mean(val_accuracies):.4f}")
print(f"  Std dev: {np.std(val_accuracies):.4f}")
print(f"  Min: {np.min(val_accuracies):.4f}")
print(f"  Max: {np.max(val_accuracies):.4f}")

print("\nTest accuracy:")
print(f"  Mean: {np.mean(test_accuracies):.4f}")
print(f"  Std dev: {np.std(test_accuracies):.4f}")
print(f"  Min: {np.min(test_accuracies):.4f}")
print(f"  Max: {np.max(test_accuracies):.4f}")

# Best model
best_idx = np.argmax(test_accuracies)
best_model = results[best_idx]

# Classification report for best model
print("\n=== Classification report for best model ===")
class_names = [f"Class {c}" for c in best_model['classes']]
print(classification_report(best_model['y_test'], best_model['y_pred'], 
                            target_names=class_names,
                            zero_division=0))

# Plot confusion matrix of best model
plt.figure(figsize=(10, 8))
cm = best_model['confusion_matrix']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=best_model['classes'], 
            yticklabels=best_model['classes'])
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.title(f'Confusion matrix - best model (exp {best_idx+1})')

# Plot average confusion matrix across all experiments
avg_cm = np.mean(confusion_matrices, axis=0)
plt.figure(figsize=(10, 8))
sns.heatmap(avg_cm, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=best_model['classes'], 
            yticklabels=best_model['classes'])
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.title('Average confusion matrix over all experiments')

# Plot class distribution (useful for checking class imbalance)
plt.figure(figsize=(10, 6))
sns.barplot(x=class_distribution.index, y=class_distribution.values)
plt.xlabel('Class')
plt.ylabel('Number of samples')
plt.title('Class distribution in dataset')

